In [3]:
import numpy as np

In [4]:
# Read header separately, then load the numeric data
with open("roe01.txt") as f:
    header = f.readline().split()

data = np.genfromtxt("roe01.txt", skip_header=1)  # shape (500, 10)
col = {name: i for i, name in enumerate(header)}

# 1. Column means
means = data.mean(axis=0)
for name, m in zip(header, means):
    print(f"{name:8s} {m:.4f}")

# 2. Correlation
corr = np.corrcoef(data, rowvar=False)
print("corr(ROEt, ROE) =", corr[col['ROEt'], col['ROE']])

# 3. Linear model: ROEt ~ ATO+PM+LEV+GROWTH+PB+ARR+INV+ASSET+ROE
x_names = ['ATO','PM','LEV','GROWTH','PB','ARR','INV','ASSET','ROE']
y = data[:, col['ROEt']]
X = data[:, [col[n] for n in x_names]]

n = X.shape[0]
X1 = np.column_stack([np.ones(n), X])          # add intercept
XtX_inv = np.linalg.inv(X1.T @ X1)
beta = XtX_inv @ X1.T @ y                       # OLS coefficients

resid = y - X1 @ beta
dof = n - X1.shape[1]
sigma2 = (resid @ resid) / dof
se = np.sqrt(np.diag(sigma2 * XtX_inv))         # standard errors
tvals = beta / se

r2 = 1 - (resid @ resid) / ((y - y.mean())**2).sum()
adj_r2 = 1 - (1 - r2) * (n - 1) / dof

for nm, b, s, t in zip(['const'] + x_names, beta, se, tvals):
    print(f"{nm:12s} coef={b:9.4f}  se={s:8.4f}  t={t:7.3f}")
print(f"R2={r2:.4f}  adj-R2={adj_r2:.4f}")

ROEt     0.0678
ATO      0.4298
PM       0.2110
LEV      0.7089
GROWTH   0.3313
PB       2.1268
ARR      0.2008
INV      0.1000
ASSET    21.0660
ROE      0.4105
corr(ROEt, ROE) = 0.5151321969256383
const        coef=  -1.3696  se=  0.5090  t= -2.691
ATO          coef=   0.0446  se=  0.0461  t=  0.967
PM           coef=   0.1636  se=  0.1290  t=  1.268
LEV          coef=  -0.0102  se=  0.0103  t= -0.991
GROWTH       coef=  -0.0075  se=  0.0094  t= -0.797
PB           coef=  -0.0018  se=  0.0034  t= -0.540
ARR          coef=   0.0121  se=  0.0236  t=  0.514
INV          coef=  -0.0526  se=  0.1632  t= -0.322
ASSET        coef=   0.0575  se=  0.0240  t=  2.395
ROE          coef=   0.4584  se=  0.0386  t= 11.875
R2=0.2879  adj-R2=0.2748
